# 92. 이러닝 영상 학습 집중도 및 완강률 분석 실습

> **📥 [실습 주피터 노트북(.ipynb) 다운로드](practice.ipynb)**
## 실전 데이터 분석 92: 온라인 강의 러닝타임 및 멈춤 횟수 대비 단원 평가 점수와 동영상 최종 완강률 분석

![도입 만화](img/intro_comic.png)

### 📊 실습 개요 (Tutorial Overview)
비대면 이러닝(E-Learning) 교육 동영상 플랫폼의 수강 학습 로그 데이터셋입니다. 강의 영상 분량(VideoDuration_Mins), 학습 중 일시 정지 멈춤 빈도(PauseCount), 학습 배속(SpeedMultiplier), 단원 평가 퀴즈 점수가 최종 이러닝 비디오 완강 비율(CompletionRate)에 미치는 시너지 효과를 분석하고 시각화합니다.

---

### 🛠️ 핵심 분석 실습 역량 (Core Skills)
* **결측치 상수 대치 (\`fillna\`):** 배속 설정 정보가 유실된 로그를 플랫폼 디폴트 시청 배속인 `1.0배속`으로 주입 정제합니다.
* **완강률 분포 요약 및 단원 퀴즈 상관성 분석 (\`histplot\`, \`scatterplot\`):** 완강률 히스토그램 및 퀴즈 평가 성적 대비 비디오 완강률의 멈춤 횟수 교차 맵을 시각화합니다.

---

## Step 1: 데이터 불러오기 및 기본 정보 확인 (Data Load)

![Step 1 데이터 수집 개념도](img/step1_dataset.svg)

제공된 CSV 파일을 분석 컴퓨터로 불러와 로드하고, 컬럼의 타입 및 데이터 구조를 확인합니다.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
import koreanize_matplotlib
sns.set_theme(style="whitegrid")

# 데이터 로드
df = pd.read_csv('./elearning_engagement.csv')
print(df.info())
print(df.head())


RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   StudentID           1000 non-null   int64  
 1   VideoDuration_Mins  1000 non-null   float64
 2   PauseCount          1000 non-null   int64  
 3   SpeedMultiplier     988 non-null    float64
 4   QuizScore           1000 non-null   int64  
 5   CompletionRate      1000 non-null   float64
dtypes: float64(3), int64(3)
memory usage: 47.0 KB
> 
> StudentID  VideoDuration_Mins  PauseCount  SpeedMultiplier  QuizScore  CompletionRate
0     920001                28.1          11             1.25         58            61.5
1     920002                21.6           0             1.00         81            87.0
2     920003                48.6           1              NaN         64            72.5
3     920004                38.4           1             1.00         66            81.2
4     920005                55.6           0             1.00         76            83.4
> ```

---

## Step 2: 결측치 및 데이터 정제 (Data Cleaning)

![Step 2 데이터 정제 개념도](img/step2_missing_values.svg)

수집 과정에서 빈칸으로 기록된 결측값(NaN)의 존재 여부를 진단하고, 데이터 도메인 성격에 부합하는 통계 수치 대입 기법으로 정제합니다.


In [ ]:
# 1. 컬럼별 결측치 개수 확인
print("--- 정제 전 결측치 확인 ---")
print(df.isnull().sum())

# 2. 결측치 전처리 및 대치 수행
df['SpeedMultiplier'] = df['SpeedMultiplier'].fillna(1.0)
print(df.isnull().sum())


VideoDuration_Mins     0
PauseCount             0
SpeedMultiplier       12
QuizScore              0
CompletionRate         0
> 
> --- 정제 후 결측치 확인 ---
> StudentID             0
VideoDuration_Mins    0
PauseCount            0
SpeedMultiplier       0
QuizScore             0
CompletionRate        0
> ```

### 💡 분석가의 통찰 (Analyst's Insight)
* **기본 정속 시청 대치의 도메인 근거:** 학습 동영상 플레이어 로그 시스템에서 사용자가 시청 속도 조절 스크롤러를 전혀 조작하지 않고 완강한 경우, 배속 이벤트 로그가 기록되지 않아 NaN으로 수집됩니다. 이 유실을 플레이어 기본 정속값인 `1.0`배속으로 채워 넣어야 수강생들의 평균 배속 편향 왜곡 없이 정밀 시청 행동을 분석할 수 있습니다.

---

## Step 3: 단변수 분포 분석 (Univariate EDA)

![Step 3 시각화 개념도](img/step3_visualization.svg)

가장 먼저 핵심 변수가 전체 데이터에서 어떤 빈도와 분포를 가졌는지 단일 변수 시각화를 통해 파악해 봅니다.


In [ ]:
plt.figure(figsize=(8, 5))

# 단변수 분석 그래프 생성
sns.histplot(data=df, x='CompletionRate', kde=True, color='purple')
plt.title('이러닝 강의 동영상 완강률(Completion Rate) 분포', fontsize=14, fontweight='bold')
plt.show()


> **💻 [실행 결과 시각화]**
> ![실행 결과 시각화](img/exec_step_3.svg)

### 💡 시각화 차트 읽는 법 & 인사이트
* **온라인 수강생들의 고관여 집중 분포 확인:** 완강률 히스토그램을 분석하면 60~95% 대역에 봉우리를 그리며 우수한 관여도를 유지하는 패턴을 띱니다. 중도 이탈자 비율은 좌측 끝단에 소수로 모여 있어 이러닝 플랫폼 강좌 구성의 우수한 품질을 입증합니다.

---

## Step 4: 다변수 상관관계 및 이상치 분석 (Multivariate EDA)

![Step 4 상관관계 분석 개념도](img/step4_multivariate.svg)

두 개 이상의 변수를 동시에 결합하여, 조건에 따른 수치 차이나 독립 변수와 종속 변수 간의 통계적 경향을 분석합니다.


In [ ]:
plt.figure(figsize=(9, 6))

# 다변수 분석 그래프 생성
sns.scatterplot(data=df, x='QuizScore', y='CompletionRate', hue='PauseCount', palette='viridis', alpha=0.8)
plt.title('퀴즈 평가 점수 대비 비디오 완강률과 멈춤 횟수 상관성', fontsize=14, fontweight='bold')
plt.show()


> **💻 [실행 결과 시각화]**
> ![실행 결과 시각화](img/exec_step_4.svg)

### 💡 코드 딥다이브 & 비즈니스 통찰 (Analyst's Insight)
* **강의 완강 관여도와 학업 성취 퀴즈 성적의 정비례 상관:** 단원 평가 퀴즈 점수(X축)와 동영상 완강률(Y축)은 매우 뚜렷한 우상향 선형 띠를 두르고 있습니다. 특히 퀴즈 고득점 구역(오른쪽 상단)에 일시정지 복습 횟수(PauseCount)가 활발했던 이력(연녹색 계열)들이 다수 오버레이되어 있어, 단순히 영상을 켜둔 것을 넘어 정지하고 뒤로 돌려보는 자기주도적 몰입 학습이 높은 학업 성취로 직결됨을 통계적 산점도로 완벽히 보장해 줍니다.

---

## Step 5: 통계적 직관과 해석 (Statistical Logic)

> 💡 **[이러닝 교육 행동 분석과 다변수 피쳐 엔지니어링의 통계 직관]**
... (생략: 학습 로그 분석 내 행동 패턴 피쳐 생성 및 효과 측정 요약)

---

## 🎯 30분 강의 마무리 및 심화 과제

오늘 우리는 실전 데이터셋을 분석하여 판다스로 데이터를 가공 및 정제하고, 시각화를 활용하여 핵심 변수 간의 통계적 유의성을 검증했습니다. 데이터 속에서 숨겨진 패턴을 올바른 시각으로 탐색하는 능력이 데이터 사이언티스트의 가장 강력한 무기입니다.

### 📝 심화 과제 (Advanced Challenge)
* **재생 속도 배속(SpeedMultiplier)과 완강률의 상관관계 분석:** 강의를 빠르게 배속으로 넘겨보는 습관이 최종 영상 이탈에 유의미한 상관을 주는지 피어슨 상관계수로 판별해 보세요.
* **자기주도 고몰입 학습자 군집 테이블 도출:** `PauseCount`가 10회 이상이고 완강률이 90%를 넘으며 퀴즈 성적이 85점 이상인 '완벽 복습형 고성과군' 수강생 리스트를 코드 추출해 보세요.
